Phase 02 : Splitting the data (Online + Periodic)

In [1]:
import pandas as pd
import hashlib

df = pd.read_csv(
    "online_retail_clean.csv",
    dtype={"transaction_id": str, "product_id": str, "customer_id": str},
    parse_dates=["event_time"],
)
print(len(df))

397884


In [3]:
def md5_number(text):
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(h[:8], 16)

In [4]:
n = df["transaction_id"].apply(md5_number)

df["sales_mode"] = "Periodic"
df.loc[n % 10 <= 6, "sales_mode"] = "Online"

In [6]:
carts = df.drop_duplicates("transaction_id")

print("Carts by mode (%):")
print(carts["sales_mode"].value_counts(normalize=True) * 100)

print("\nCarts by mode:")
print(carts["sales_mode"].value_counts())

print("\nRows by mode:")
print(df["sales_mode"].value_counts())

# every cart must have exactly one sales_mode
print("\nCarts with mixed mode:",
      (df.groupby("transaction_id")["sales_mode"].nunique() > 1).sum())

Carts by mode (%):
sales_mode
Online      69.992445
Periodic    30.007555
Name: proportion, dtype: float64

Carts by mode:
sales_mode
Online      12971
Periodic     5561
Name: count, dtype: int64

Rows by mode:
sales_mode
Online      278740
Periodic    119144
Name: count, dtype: int64

Carts with mixed mode: 0


In [7]:
df.to_csv("online_retail_split.csv", index=False)
print("Saved:", len(df), "rows")

Saved: 397884 rows
